# Calculating monthly mean MDA8: 8hr Daily Maximum

Input data should be in the format: **hourly surface ozone in ppb** coordinates: lat, lon, time (1hr)

Output data will be in the format: **monthly ozone (MDA8) in ppb** coordinates: lat, lon, time (month)

In [ ]:
import os
import xarray as xr
import numpy as np
import glob
from utils.utils import get_scenario_config, standardise_latlon
import config
from utils.utils import require_dir
import pathlib

In [ ]:
# === Processing function ===
def calculate_monthly_mean_8hrdailymax(start_date, end_date, o3_surf, calendar):
    daterange = xr.date_range(start_date, end_date, calendar=calendar, use_cftime=True)

    MDA8 = xr.DataArray(  # Maximum Daily 8hr Average O3
        np.nan,
        dims=["time", "lat", "lon"],
        coords={"time": daterange, "lat": o3_surf.lat, "lon": o3_surf.lon},
    )

    # Loop through each day
    for i in range(len(daterange)):
        date = daterange[i].strftime("%Y-%m-%d")
        # print(f"Processing {date}")
        o3_day = o3_surf.sel(time=slice(date + " 00:00:00", date + " 23:00:00"))

        o3_rolling = o3_day.rolling(time=8).mean()
        MDA8[i, :, :] = o3_rolling.max("time")

    monthly_mean = MDA8.resample(time="ME").mean()
    return monthly_mean

In [ ]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245"

configs = get_scenario_config(model, scenario)
ensemble_members = configs["ensemble_members"]
years = configs["years"]

FILE_DIR = require_dir(pathlib.Path(config.SCRATCH_ROOT) / model / "ozone" / "hourly_o3")
SAVE_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "ozone" / "MDA8")

# === Main loop ===
for ens_num in ensemble_members:
    print(f"Processing {scenario}, Ensemble {ens_num:02d}")
    # Create file list
    file_pattern = os.path.join(FILE_DIR, f"Hourly_surface_o3_{model}_{scenario}_{ens_num:02d}_*.nc")
    file_list = sorted(glob.glob(file_pattern))
    monthly_means = []

    for f in file_list:
        if not os.path.exists(f):
            raise ValueError(f"Missing: {f}")

        print(f"Reading {os.path.basename(f)}")
        da = standardise_latlon(xr.open_dataarray(f))

        start_date = str(da.time[0].values)[:10]  # e.g. yyyy-mm-dd
        end_date = str(da.time[-1].values)[:10]
        end_time = str(da.time[-1].values)[11:16]  # e.g. hh:mm
        midnight = "00:00"

        if end_time == midnight:
            print("changing final time step")
            # making final time step 23:00
            end_date = str(da.time[-2].values)[:10]

        # Different climate models use different calendar types
        model_calendar = da.time.encoding.get("calendar")

        mm = calculate_monthly_mean_8hrdailymax(
            start_date,
            end_date,
            da,
            model_calendar)

        monthly_means.append(mm)

    if monthly_means:
        combined = xr.concat(monthly_means, dim="time")

        time = combined.indexes["time"]
        if time.freq is None:
            combined = combined.sortby("time")

        # Trim to start_year - end_year
        combined = combined.sel(time=slice(str(years.start), str(years.stop)))

        # Create date stamp for file saving
        start_time = combined["time"][0].item().strftime("%Y%m")
        end_time = combined["time"][-1].item().strftime("%Y%m")
        dates = f"{start_time}-{end_time}"

        out_file = f"MDA8_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving to {out_path}")
        description = ("Monthly MDA8: 8hr Daily Maximum. Calculates the 8-hour "
                       "daily maximum surface ozone concentration - scripts "
                       "by A.F. Wells (2025)")
        combined.attrs["description"] = description
        combined.attrs["units"] = "ppb"
        combined.attrs["ensemble_number"] = ens_num
        combined.attrs["scenario"] = scenario
        combined.attrs["model"] = model
        combined.to_netcdf(out_path)

print("All processing complete.")
